# neural density estimation --- normalizing flows
We'll implement a few normalizing flows using `pytorch`

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import numpy as np 
import matplotlib.pyplot as plt
import corner

In [ ]:
torch.manual_seed(0)

In [ ]:
if torch.cuda.is_available(): 
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else: 
    device = torch.device("cpu")
print("Using device:", device)

# target distribution $p(x)$
Lets use the same gnarly distirbution as in the MADE example

In [ ]:
def make_synthetic_data(N, device=device):
    ''' complicated 4D distribution: 
    - (x0, x1): 3-component Gaussian mixture with different means & covariances
    - x2: nonlinear function of (x0, x1) + noise
          x2 = sin(x0) + 0.5 * x1^2 + eps
    - x3: heavy-tailed (Student-t-like) variable
    '''
    mus = torch.stack([
        torch.tensor([0.0, 0.0], device=device),
        torch.tensor([3.0, -3.0], device=device),
        torch.tensor([-3.0, 3.0], device=device)]) 

    covs = []
    covs.append(torch.tensor([[1.0, 0.2],
                              [0.2, 1.0]], device=device))
    covs.append(torch.tensor([[2.0, 1.5],
                              [1.5, 2.0]], device=device))
    covs.append(torch.tensor([[0.5, -0.3],
                              [-0.3, 0.5]], device=device))
    covs = torch.stack(covs, dim=0)  
    pis = torch.tensor([0.4, 0.35, 0.25], device=device)

    comp_idx = torch.multinomial(pis, num_samples=N, replacement=True)  # [N]

    # For each sample, get its mean and covariance
    mu = mus[comp_idx]           # [N, 2]
    cov = covs[comp_idx]         # [N, 2, 2]

    # Sample from N(0, I) and transform via Cholesky of cov
    eps = torch.randn(N, 2, device=device)  # [N, 2]
    # Cholesky factor per component
    L = torch.linalg.cholesky(cov)          # [N, 2, 2]
    xy = (L @ eps.unsqueeze(-1)).squeeze(-1) + mu  # [N, 2]

    x0 = xy[:, 0]
    x1 = xy[:, 1]

    # ----- Nonlinear, noisy x2 -----
    # x2 = sin(x0) + 0.5 * x1^2 + noise
    noise_x2 = 0.5 * torch.randn(N, device=device)
    x2 = torch.sin(x0) + 0.5 * x1**2 + noise_x2

    # ----- Heavy-tailed x3 (Student-t like) -----
    # t-dist ~ z / sqrt(u/ν), with z ~ N(0,1), u ~ χ²(ν)
    nu = 3.0
    z = torch.randn(N, device=device)
    u = torch.distributions.Chi2(df=torch.tensor(nu, device=device)).sample((N,))
    x3 = z * torch.sqrt(nu / u)  # Student-t(ν)

    return torch.stack([x0, x1, x2, x3], dim=-1)

In [ ]:
samples = make_synthetic_data(10000, device='cpu')
fig = plt.figure(figsize=(6,6))
fig = corner.corner(samples.numpy(), range=[(-5, 8), (-7, 5), (-5, 15), (-10, 10)], fig=fig)

# Planar flow
Lets start with one of the simplest normalizing flow: the planar flow:
$$x = z + \hat{u}\tanh(w^\top z + b)$$
where $u$, $w$, and $b$ are learnable parameters.


To fit a flow to data, we need $\log p(x)$:
$$z = f^{-1}(x) = x + \hat{u}\tanh(w^\top x + b), \qquad \log p(x) = \log\pi(z) + \log\left|1 + \hat{u}^\top h'(w^\top x + b)\,w\right|.$$
Evaluating $\log p(x)$ is then exact and cheap. 

Note, we use $\hat{u}$, a reparameterization of $u$, to ensure invertibility. 
Raw $u$ is unconstrained but $w^T\hat{u} \ge -1$ must hold. 
So we use the reparameterization
$$\hat{u} = u + ({\rm softplus}(w^Tu)-w^Tu-1)\frac{w}{||w||^2}$$
which gives $w^\top\hat{u} = {\rm softplus}(w^\top u) - 1 > -1$.

In [ ]:
class PlanarFlow(nn.Module):
    ''' one planar layer, used in the normalizing direction: forward maps x -> z '''
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

        self.u = nn.Parameter(torch.zeros(dim))
        self.w = nn.Parameter(torch.zeros(dim))
        self.b = nn.Parameter(torch.zeros(()))

        # initialize
        nn.init.normal_(self.u, mean=0.0, std=0.1)
        nn.init.normal_(self.w, mean=0.0, std=0.1)
        nn.init.zeros_(self.b)

    def _u_hat(self):
        ''' Reparameterization of u to enforce invertibility: w^T u_hat = softplus(w^T u) - 1 > -1
        '''
        wtu = torch.dot(self.w, self.u)  # scalar
        w_norm_sq = torch.sum(self.w ** 2)
        u_hat = self.u + (F.softplus(wtu) - wtu - 1.0) * self.w / (w_norm_sq + 1e-8)
        return u_hat

    def forward(self, x):
        ''' x -> z = x + u_hat tanh(w^T x + b), and log |det dz/dx|
        '''
        u_hat = self._u_hat()
        linear = x @ self.w + self.b
        z = x + torch.tanh(linear).unsqueeze(1) * u_hat

        # psi(x) = h'(linear) * w, where h' = 1 - tanh^2
        psi = (1.0 - torch.tanh(linear) ** 2).unsqueeze(1) * self.w

        # log |1 + u_hat^T psi|
        log_det = torch.log(torch.abs(1.0 + psi @ u_hat) + 1e-8)
        return z, log_det

    @torch.no_grad()
    def inverse(self, z, n_iter=60):
        ''' z -> x. 
        '''
        u_hat = self._u_hat()
        wtu = torch.dot(self.w, u_hat)
        wtz = z @ self.w
        lo, hi = wtz - wtu.abs() - 1e-3, wtz + wtu.abs() + 1e-3 
        for _ in range(n_iter):
            a = 0.5 * (lo + hi)
            too_big = a + wtu * torch.tanh(a + self.b) > wtz
            hi = torch.where(too_big, a, hi)
            lo = torch.where(too_big, lo, a)
        a = 0.5 * (lo + hi)
        return z - torch.tanh(a + self.b).unsqueeze(1) * u_hat

    def log_prob(self, x):
        ''' log p(x) = log N(z; 0, I) + log |det dz/dx|, 
        '''
        z, log_det = self.forward(x)
        return -0.5 * (self.dim * np.log(2 * np.pi) + torch.sum(z ** 2, dim=1)) + log_det

    @torch.no_grad()
    def sample(self, num_samples):
        ''' smaple z' ~ N(z; 0, I) then z' -> x' (inverse)
        '''
        z = torch.randn(num_samples, self.dim, device=self.w.device)
        return self.inverse(z)

In [ ]:
batch_size = 256
num_epochs = 100

flow = PlanarFlow(4).to(device)

optimizer = torch.optim.Adam(flow.parameters(), lr=1e-2)

# Data
x_train = samples.to(device) 
N_train = samples.shape[0]

num_batches = N_train // batch_size
train_losses = []
for epoch in range(1, num_epochs + 1):
    # Shuffle train data
    perm = torch.randperm(N_train, device=device)
    x_train = x_train[perm]

    flow.train()
    epoch_loss = 0.0

    for i in range(num_batches):
        batch_x = x_train[i * batch_size:(i + 1) * batch_size]
        optimizer.zero_grad()
        lp = flow.log_prob(batch_x)      
        loss = -lp.mean()                
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    epoch_loss /= num_batches
    train_losses.append(epoch_loss)

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:03d}: train -log p = {epoch_loss:.4f}")

again, for the purposes of this demo we won't go through validation but always validate!

Let's see how well the flow estimates the distribution. To sample, draw $z \sim \mathcal{N}(0, I)$ and **invert** the
flow, $x = f(z)$.

In [ ]:
flow.eval()
q_samples = flow.sample(10000).cpu()

In [ ]:
fig = plt.figure(figsize=(6,6))
fig = corner.corner(samples.numpy(), hist_kwargs={'density': True}, 
                    range=[(-5, 8), (-7, 5), (-5, 15), (-10, 10)], fig=fig)
fig = corner.corner(q_samples.numpy(), color='C1', hist_kwargs={'density': True}, 
                    range=[(-5, 8), (-7, 5), (-5, 15), (-10, 10)], fig=fig)

Yikes. Now that's not a terribly good fit to the distribution: a single planar layer can only bend the Gaussian along one direction, $w$, and its loss stalls at $-\log p \approx 13.5$. But that was with a single flow.

In [ ]:
n_param = sum(p.numel() for p in flow.parameters())
print('%i free parameters in the flow' % n_param)

The single planar flow only has 9 free parameters ($u$, $w$ and $b$). The major advantage of flows is that you can stack multiple of them. Lets build a normalizing flow that's a combination of multiple planar flows.

# Flows 2: Planar Flow**S** 

In [ ]:
class PlanarFlows(nn.Module):
    def __init__(self, dim, num_flows):
        super().__init__()
        self.dim = dim
        self.flows = nn.ModuleList([PlanarFlow(dim) for _ in range(num_flows)])

    def forward(self, x):
        ''' x -> z through every layer; the log-determinants add '''
        z = x
        sum_log_det = torch.zeros(z.size(0), device=z.device)

        for flow in self.flows:
            z, log_det = flow(z)
            sum_log_det += log_det

        log_p0 = -0.5 * (self.dim * np.log(2 * np.pi) + torch.sum(z ** 2, dim=1))
        log_px = log_p0 + sum_log_det
        return z, log_px

    def log_prob(self, x):
        _, log_px = self.forward(x)
        return log_px

    @torch.no_grad()
    def sample(self, num_samples):
        ''' z ~ N(0, I), then invert the layers in reverse order '''
        x = torch.randn(num_samples, self.dim, device=self.flows[0].w.device)
        for flow in reversed(self.flows):
            x = flow.inverse(x)
        return x

In [ ]:
batch_size = 256
num_epochs = 100

N_flows = 16
flows = PlanarFlows(4, N_flows).to(device)

optimizer = torch.optim.Adam(flows.parameters(), lr=1e-2)

# Data
x_train = samples.to(device) 
N_train = samples.shape[0]

num_batches = N_train // batch_size
train_losses = []
for epoch in range(1, num_epochs + 1):
    # Shuffle train data
    perm = torch.randperm(N_train, device=device)
    x_train = x_train[perm]

    flows.train()
    epoch_loss = 0.0

    for i in range(num_batches):
        batch_x = x_train[i * batch_size:(i + 1) * batch_size]
        optimizer.zero_grad()
        lp = flows.log_prob(batch_x)      # [batch]
        loss = -lp.mean()                  # NLL
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    epoch_loss /= num_batches
    train_losses.append(epoch_loss)

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:03d}: train -log p = {epoch_loss:.4f}")

In [ ]:
flows.eval()
q_samples = flows.sample(10000).cpu()

In [ ]:
fig = plt.figure(figsize=(6,6))
fig = corner.corner(samples.numpy(), hist_kwargs={'density': True}, 
                    range=[(-5, 8), (-7, 5), (-5, 15), (-10, 10)], fig=fig)
fig = corner.corner(q_samples.numpy(), color='C1', hist_kwargs={'density': True}, 
                    range=[(-5, 8), (-7, 5), (-5, 15), (-10, 10)], fig=fig)

Much better! With 16 layers there are $16 \times 9 = 144$ parameters. 

# Masked Autoregressive Flows

In [ ]:
def create_masks(hidden_dims, ndim=4, device="cpu"):
    # indices for input layer
    input_indices = torch.arange(1, ndim + 1, device=device)

    # indices for hidden layers in [1, ndim - 1]
    indices = []
    for h in hidden_dims:
        # for each hidden unit, sample an integer index in [1, ndim - 1]
        ind = torch.randint(1, ndim, size=(h,), device=device)
        indices.append(ind)

    output_indices = torch.arange(1, ndim + 1, device=device).repeat_interleave(2)

    # ndim-dimensional input --> mu, \log sigma per dimension
    layer_sizes = [ndim] + hidden_dims + [2 * ndim]
    all_indices = [input_indices] + indices + [output_indices]

    masks = []
    for l in range(len(layer_sizes) - 1):
        i_in = all_indices[l]      # [n_in]
        i_out = all_indices[l + 1] # [n_out]
        if l < len(layer_sizes) - 2:
            # hidden layers: allow if d_out >= d_in
            mask = (i_out[:, None] >= i_in[None, :]).float()
        else:
            # output layer: allow if d_out > d_in
            mask = (i_out[:, None] > i_in[None, :]).float()
        masks.append(mask)
    return masks
    
# same MADE class object as the MADE example
class MADE(nn.Module):
    def __init__(self, hidden_dims, D=4, device="cpu"):
        super().__init__()
        self.D = D
        self.hidden_dims = hidden_dims
        self.device = device

        layer_sizes = [D] + hidden_dims + [D * 2]

        layers = []
        for in_f, out_f in zip(layer_sizes[:-1], layer_sizes[1:]):
            layers.append(nn.Linear(in_f, out_f))
        self.layers = nn.ModuleList(layers)

        # build masks
        masks = create_masks(hidden_dims, ndim=D, device=self.device)
        # so they move with .to(device), but are not trainable
        self.masks = nn.ParameterList(
            [nn.Parameter(m, requires_grad=False) for m in masks]
        )
        
        # initialize W and b
        for layer in self.layers:
            nn.init.normal_(layer.weight, mean=0.0, std=1.0 / np.sqrt(layer.in_features))
            nn.init.zeros_(layer.bias)
            
        # near-identity init: last layer outputs ~0
        nn.init.zeros_(self.layers[-1].weight)
        nn.init.zeros_(self.layers[-1].bias)

    def forward(self, x):
        '''forward pass 
        '''
        h = x
        L = len(self.layers)
        for l, layer in enumerate(self.layers):
            # apply autoregressive mask to weights
            masked_weight = layer.weight * self.masks[l]
            h = F.linear(h, masked_weight, layer.bias)
            if l < L - 1:
                h = F.relu(h)
                
        B = h.shape[0]
        h = h.view(B, self.D, 2)
        mean = h[..., 0]
        log_std = h[..., 1]
        return mean, log_std

    def log_prob(self, x):
        ''' log p(x)
        '''
        mean, log_std = self.forward(x)
        inv_std = torch.exp(-log_std)
        z = (x - mean) * inv_std
        log_det = -log_std
        logp_per_dim = -0.5 * (np.log(2.0 * np.pi) + z**2) + log_det
        return torch.sum(logp_per_dim, dim=-1)

In [ ]:
class MAFBlock(nn.Module):
    '''single MAF block'''
    def __init__(self, D, hidden_dims, device="cpu"):
        super().__init__()
        self.D = D
        self.net = MADE(hidden_dims, D=D, device=device)

        perm = torch.randperm(D)
        self.register_buffer("perm", perm)
        self.register_buffer("inv_perm", torch.argsort(perm))

    def forward(self, x):
        ''' x -> z 
        '''
        x_perm = x[:, self.perm]

        m, log_s = self.net(x_perm)
        log_s = torch.clamp(log_s, -5, 5)          # prevent exp explosion

        z_perm = (x_perm - m) * torch.exp(-log_s)
        log_det = -log_s.sum(dim=1)                 # sum over D dimensions

        return z_perm[:, self.inv_perm], log_det

    def inverse(self, z):
        ''' z -> x
        '''
        z_perm = z[:, self.perm]
        x_perm = torch.zeros_like(z_perm)

        for i in range(self.D):
            m, log_s = self.net(x_perm)
            log_s = torch.clamp(log_s, -5, 5)
            x_perm[:, i] = z_perm[:, i] * torch.exp(log_s[:, i]) + m[:, i]

        return x_perm[:, self.inv_perm]


class MaskedAutoregressiveFlow(nn.Module):
    ''' MAF with K MAFBlocks and a standard Normal base distribution.
    '''
    def __init__(self, D, hidden_dims, num_flows, device="cpu"):
        super().__init__()
        self.D = D
        self.device = device
        self.flows = nn.ModuleList(
            [MAFBlock(D, hidden_dims, device=device) for _ in range(num_flows)]
        )

    def forward(self, x):
        # x -> z
        z = x
        sum_log_det = torch.zeros(z.size(0), device=z.device)
        for flow in self.flows:
            z, log_det = flow(z)
            sum_log_det += log_det
        return z, sum_log_det

    def log_prob(self, x):
        z, sum_log_det = self.forward(x)
        log_p0 = -0.5 * (self.D * np.log(2 * np.pi) + (z ** 2).sum(dim=1))
        return log_p0 + sum_log_det

    @torch.no_grad()
    def sample(self, num_samples):
        ''' sample z ~ N(0,1) then invert'''
        self.eval()
        z = torch.randn(num_samples, self.D, device=self.device)
        for flow in reversed(self.flows):           # invert in reverse order
            z = flow.inverse(z)
        return z

In [ ]:
hidden_dims = [128, 128]
num_flows   = 8
batch_size  = 256
num_epochs  = 150

maf = MaskedAutoregressiveFlow(4, hidden_dims, num_flows, device=str(device)).to(device)
optimizer = torch.optim.Adam(maf.parameters(), lr=1e-3)

x_train     = samples.to(device)
N_train     = x_train.shape[0]
num_batches = N_train // batch_size

train_losses = []
for epoch in range(1, num_epochs + 1):
    maf.train()
    x_train = x_train[torch.randperm(N_train, device=device)]
    epoch_loss = 0.0
    for i in range(num_batches):
        batch = x_train[i * batch_size:(i + 1) * batch_size]
        optimizer.zero_grad()
        loss = -maf.log_prob(batch).mean()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        
    epoch_loss /= num_batches
    train_losses.append(epoch_loss)
    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch {epoch:03d} | log p: {epoch_loss:.4f}")

In [ ]:
maf.eval()
with torch.no_grad():
    q_samples = maf.sample(10000).cpu()

In [ ]:
fig = plt.figure(figsize=(6,6))
fig = corner.corner(samples.numpy(), hist_kwargs={'density': True}, 
                    range=[(-5, 8), (-7, 5), (-5, 15), (-10, 10)], fig=fig)
fig = corner.corner(q_samples.numpy(), color='C1', hist_kwargs={'density': True}, 
                    range=[(-5, 8), (-7, 5), (-5, 15), (-10, 10)], fig=fig)

Even better than planar flow

# In-class exercise: the galaxy color bimodality

Galaxies come in two populations: star-forming galaxies in a **blue cloud** and quiescent galaxies on a **red sequence**. 
Plotted as color against stellar mass, a galaxy catalog is strongly bimodal: at fixed mass, the distribution of color has two peaks. 
Let's learn the density $p(\log M_*, u-r)$ of a galaxy catalog with a normalizing flow.

The catalog is simulated, so we know the true density and can check the flow exactly:

- **Red sequence** (45% of galaxies): $\log M_* \sim \mathcal{N}(10.6, 0.35^2)$, $u-r = 2.45 + 0.08(\log M_* - 10)$ with
  scatter 0.07.
- **Blue cloud** (55%): $\log M_* \sim \mathcal{N}(9.9, 0.45^2)$, $u-r = 1.45 + 0.25(\log M_* - 10)$ with scatter 0.15.

We'll compare four density estimators, all fit using the same 10,000 training galaxies:

1. a single Gaussian;
2. a single MADE (Lecture 21)
4. a MAF with several blocks

Reuse `MADE`, `MAFBlock` and `MaskedAutoregressiveFlow` from the demo above

In [ ]:
from scipy.stats import multivariate_normal, norm
import time

dev = "cpu"
torch.manual_seed(22)
rng = np.random.default_rng(22)

f_red = 0.45
red  = dict(mu_m=10.6, s_m=0.35, a=2.45, b=0.08, s_c=0.07)   # red sequence
blue = dict(mu_m=9.9,  s_m=0.45, a=1.45, b=0.25, s_c=0.15)   # blue cloud

def simulate_catalog(N):
    """ N galaxies: columns (log M*, u-r) """
    is_red = rng.random(N) < f_red
    logM = np.where(is_red, rng.normal(red['mu_m'], red['s_m'], N), rng.normal(blue['mu_m'], blue['s_m'], N))
    color = np.where(is_red,
                     red['a'] + red['b'] * (logM - 10) + rng.normal(0, red['s_c'], N),
                     blue['a'] + blue['b'] * (logM - 10) + rng.normal(0, blue['s_c'], N))
    return np.stack([logM, color], axis=1)

def mean_cov(p):
    """ each population is a 2D Gaussian, because u-r is linear in log M* """
    mean = np.array([p['mu_m'], p['a'] + p['b'] * (p['mu_m'] - 10)])
    cov = np.array([[p['s_m']**2,            p['b'] * p['s_m']**2],
                    [p['b'] * p['s_m']**2,   p['b']**2 * p['s_m']**2 + p['s_c']**2]])
    return mean, cov

def log_p_true(X):
    """ exact log density of the catalog """
    return np.logaddexp(np.log(f_red)     + multivariate_normal(*mean_cov(red)).logpdf(X),
                        np.log(1 - f_red) + multivariate_normal(*mean_cov(blue)).logpdf(X))

X_train = simulate_catalog(10000)
X_valid = simulate_catalog(2500)
X_test  = simulate_catalog(10000)

In [ ]:
lims = [(8.3, 11.8), (0.6, 3.0)]
fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(X_train[:, 0], X_train[:, 1], s=1, c='k', alpha=0.3)
ax.set_xlim(lims[0]); ax.set_ylim(lims[1])
ax.set_xlabel(r'$\log M_*\ [M_\odot]$'); ax.set_ylabel(r'$u - r$')
ax.set_title('10,000 training galaxies');

## 1. Gaussian surrogate and a single MADE
Train a 2D Gaussian surrogate and a single MADE to approximate $p(\log M_*, u-r)$. 

Don't forget to standardize/whiten the data first, implement early stopping, and validate the loss curves as always. 

## 2. Train flows
Follow the demo above to train a `PlanarFlows` and a `MaskedAutogressiveFlow` to approximate $p(\log M_*, u-r)$. Use 4 blocks. How do the validation losses compare between the flows? 

*Experiment with different numbers of blocks. How does the validation loss change?*

## 3. Compare the surrogates

Since we can evaluate the surrogates at a specific ${\bf x}$, $q_\phi({\bf x}$ (or log_q), we can look at conditional distributions of color at fixed $\log M_\star$ --- i.e. $q(u-r \mid \log M_* = 10.2)$. 
Compare $q(u-r \mid \log M_* = 10.2)$ of the three surrogates. 

*Hint: you can evaluate $q$ on a grid of colors at fixed mass and normalize it numerically OR you can sample the distribution using MCMC.*

Sample each of the surrogates and compare against the true distribution in the $u-r$ vs $\log M_*$ space.

## 4. Checking the flow
A flow maps $z \rightarrow x$ where $z \sim \mathcal{N}(0, I)$. Let's check that mapping. 
Use the MAF to map the 10,000 observed galaxies to $z$-space and plot the distribution using `corner`. Compare it to a unit Gaussian.

## (Optional) 500 galaxies
Try training a MAF with different number of blocks on 500 galaxies. 
